In [ ]:
# Replace the following with your path
%cd "/path/to/your/folder"

In [ ]:
# Make a function that retreives the most recent record
def get_trial_results(nct_id):
    url = f"https://clinicaltrials.gov/api/v2/studies/{nct_id}?format=json"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to retrieve data for {nct_id}")
        return None

In [ ]:
# Some CT.gov fields may be useful for defining certain outcome elements but not others, 
# so we created separate downloading code for each element.

In [ ]:
# 1. Specifc measurement

import requests  
import json  
from tqdm import tqdm  
import os  
import pandas as pd
    
# Read NCT IDs
with open('nctid_new150.txt', 'r') as file:
    nct_ids = file.read().splitlines()

output_dir = 'out_last_meas' # Specify the input file path and output directory

os.makedirs(output_dir, exist_ok=True) # Create the output directory if it doesn't exist

def save_to_json(nct_id, data):
    output_file = os.path.join(output_dir, f"{nct_id}.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

for nct_id in tqdm(nct_ids, desc="Processing NCT IDs"):  
    result = get_trial_results(nct_id)  
    if result:  
        try:  
            nct_data = []  
            study_protocol_section = result.get('resultsSection', {})  
            if 'outcomeMeasuresModule' in study_protocol_section:  
                outcome_measures = study_protocol_section['outcomeMeasuresModule'].get('outcomeMeasures', [])  
                outcome_counters = {'PRIMARY': 0, 'SECONDARY': 0, 'OTHER': 0, 'POST_HOC': 0, "OTHER_PRE_SPECIFIED": 0}  
                for measure in outcome_measures:  
                    priority = measure.get('type', '').upper()  
                    if priority in outcome_counters:  
                        outcome_counters[priority] += 1  
                    outcome_number = outcome_counters[priority]  
                    identifier = f"{priority.capitalize()} {outcome_number}"  
                    title = measure.get('title', '')  
                    description = measure.get('description', '')  
                    time_frame = measure.get('timeFrame', '')  
                    param_type = measure.get('paramType', '')  
                    dispersion_type = measure.get('dispersionType', '')  
                    unit = measure.get('unitOfMeasure', '')  
  
                    # Extracting titles from classes and converting to string
                    cate = measure.get('classes', [])  
                    titles = [cat.get("title", "") for cat in cate]  
                    titles_str = "; ".join(titles)  # Convert list of titles to a single string  
  
                     #denom = measure.get('denoms', [])
#                     if denom:
#                         total_count = float(denom[0]['counts'][0]['value'])  # Convert total_count to float

#                     def safe_float(value):
#                         if value == 'NA':  # Handle missing data
#                             return float('nan')  # Or return None if preferred
#                         try:
#                             return float(value.replace(',', ''))  # Remove commas before conversion
#                         except ValueError:
#                             raise ValueError(f"Unable to convert value to float: {value}")

                    # Then modify your list comprehension:

#                     first_values = [safe_float(item['categories'][0]['measurements'][0]['value']) for item in cate]
                        
#                     if not first_values or total_count is None:
#                         single_categorization = ""
#                         group_count = ""
#                     else:
#                         group_count = "; ".join(map(str, first_values))
#                         if sum(first_values) == total_count:
#                             single_categorization = "yes"
#                         else:
#                             single_categorization = "no"

                            
                    analyses = measure.get('analyses', [])  
                    if analyses:  
                        for analysis in analyses:  
                            method = analysis.get('statisticalMethod', '')  
                            a_param_type = analysis.get('paramType', '')  
  
                            outcome_data = {  
                                "Identifier": identifier,  
                                "Outcome measure": title,  
                                "Outcome description": description,  
                                "Time frame": time_frame,  
                                #"Measure type": param_type,  
                                #"Measure of Dispersion/Precision": dispersion_type,  
                                #"Unit of Measure": unit,  
                                #"Statistical Analysis Method": method,  
                                #"Outcome Analysis Parameter Type": a_param_type,  
                                "Classes": titles_str  # Use the string of titles here  
                                #"Single Categorization": single_categorization
                            }  
                            nct_data.append(outcome_data)  
                    else:  
                        outcome_data = {  
                            "Identifier": identifier,  
                            "Outcome measure": title,  
                            "Outcome description": description,  
                            "Time frame": time_frame,  
                            #"Measure type": param_type,  
                            #"Measure of Dispersion/Precision": dispersion_type,  
                            #"Unit of Measure": unit,  
                            #"Statistical Analysis Method": "",  
                            #"Outcome Analysis Parameter Type": "",  
                            "Classes": titles_str  # Use the string of titles here  
                            #"Single Categorization": single_categorization
                        }  
                        nct_data.append(outcome_data)  
            else:  
                print(f"No Outcome Measures Module found for {nct_id}")  
                nct_data.append({"error": "No Outcome Measures Module found"})  
  
            save_to_json(nct_id, nct_data)  
        except KeyError as e:  
            print(f"Error navigating JSON structure for {nct_id}: {e}")  
            save_to_json(nct_id, [{"error": f"Error navigating JSON structure: {e}"}])  
print("Data extraction complete. Results saved to individual JSON files.")  



In [ ]:
# 2. Specifc metric

import requests  
import json  
from tqdm import tqdm  
import os  
import pandas as pd
    
# Read NCT IDs
with open('nctid_new150.txt', 'r') as file:
    nct_ids = file.read().splitlines()

output_dir = 'out_last_metric' # Specify the input file path and output directory

os.makedirs(output_dir, exist_ok=True) # Create the output directory if it doesn't exist

def save_to_json(nct_id, data):
    output_file = os.path.join(output_dir, f"{nct_id}.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

for nct_id in tqdm(nct_ids, desc="Processing NCT IDs"):  
    result = get_trial_results(nct_id)  
    if result:  
        try:  
            nct_data = []  
            study_protocol_section = result.get('resultsSection', {})  
            if 'outcomeMeasuresModule' in study_protocol_section:  
                outcome_measures = study_protocol_section['outcomeMeasuresModule'].get('outcomeMeasures', [])  
                outcome_counters = {'PRIMARY': 0, 'SECONDARY': 0, 'OTHER': 0, 'POST_HOC': 0, "OTHER_PRE_SPECIFIED": 0}  
                for measure in outcome_measures:  
                    priority = measure.get('type', '').upper()  
                    if priority in outcome_counters:  
                        outcome_counters[priority] += 1  
                    outcome_number = outcome_counters[priority]  
                    identifier = f"{priority.capitalize()} {outcome_number}"  
                    title = measure.get('title', '')  
                    description = measure.get('description', '')  
                    time_frame = measure.get('timeFrame', '')  
                    param_type = measure.get('paramType', '')  
                    dispersion_type = measure.get('dispersionType', '')  
                    unit = measure.get('unitOfMeasure', '')  
  
                    # Extracting titles from classes and converting to string
                    cate = measure.get('classes', [])  
                    titles = [cat.get("title", "") for cat in cate]  
                    titles_str = "; ".join(titles)  # Convert list of titles to a single string  
  
                    denom = measure.get('denoms', [])
                    if denom:
                        total_count = float(denom[0]['counts'][0]['value'])  # Convert total_count to float
                        
                    def safe_float(value):
                        if value == 'NA':
                            return float('nan')  # or you could return None, or any other value that represents missing data in your context
                        return float(value)

                    # Then modify your list comprehension:
                    #first_values = [safe_float(item['categories'][0]['measurements'][0]['value']) for item in cate]
                    #group_count = "; ".join(map(str, first_values))  # Convert list of values to a single string

                    # Calculate the sum of first_values and compare with total_count
                    #if sum(first_values) == total_count and (~pd.isna(total_count)) and (group_count!="") and (~pd.isna(sum(first_values))):
                   #     single_categorization = "yes"
                    #else:
                     #   single_categorization = "no"
                            
                    analyses = measure.get('analyses', [])  
                    if analyses:  
                        for analysis in analyses:  
                            method = analysis.get('statisticalMethod', '')  
                            a_param_type = analysis.get('paramType', '')  
  
                            outcome_data = {  
                                "Identifier": identifier,  
                                "Outcome measure": title,  
                                "Outcome description": description,  
                                "Time frame": time_frame,  
                                #"Measure type": param_type,  
                                #"Measure of Dispersion/Precision": dispersion_type,  
                                "Unit of Measure": unit,  
                                "Statistical Analysis Method": method,  
                                "Outcome Analysis Parameter Type": a_param_type,  
                                "Classes": titles_str  # Use the string of titles here  
                                #"Single Categorization": single_categorization
                            }  
                            nct_data.append(outcome_data)  
                    else:  
                        outcome_data = {  
                            "Identifier": identifier,  
                            "Outcome measure": title,  
                            "Outcome description": description,  
                            "Time frame": time_frame,  
                            #"Measure type": param_type,  
                            #"Measure of Dispersion/Precision": dispersion_type,  
                            "Unit of Measure": unit,  
                            "Statistical Analysis Method": "",  
                            "Outcome Analysis Parameter Type": "",  
                            "Classes": titles_str  # Use the string of titles here  
                            #"Single Categorization": single_categorization
                        }  
                        nct_data.append(outcome_data)  
            else:  
                print(f"No Outcome Measures Module found for {nct_id}")  
                nct_data.append({"error": "No Outcome Measures Module found"})  
  
            save_to_json(nct_id, nct_data)  
        except KeyError as e:  
            print(f"Error navigating JSON structure for {nct_id}: {e}")  
            save_to_json(nct_id, [{"error": f"Error navigating JSON structure: {e}"}])  
print("Data extraction complete. Results saved to individual JSON files.")  



In [ ]:
# 3. Cutoff (same fields as Specifc measurement)

import requests  
import json  
from tqdm import tqdm  
import os  
import pandas as pd
    
# Read NCT IDs
with open('nctid_new150.txt', 'r') as file:
    nct_ids = file.read().splitlines()

output_dir = 'out_last_cut' # Specify the input file path and output directory

os.makedirs(output_dir, exist_ok=True) # Create the output directory if it doesn't exist

def save_to_json(nct_id, data):
    output_file = os.path.join(output_dir, f"{nct_id}.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

for nct_id in tqdm(nct_ids, desc="Processing NCT IDs"):  
    result = get_trial_results(nct_id)  
    if result:  
        try:  
            nct_data = []  
            study_protocol_section = result.get('resultsSection', {})  
            if 'outcomeMeasuresModule' in study_protocol_section:  
                outcome_measures = study_protocol_section['outcomeMeasuresModule'].get('outcomeMeasures', [])  
                outcome_counters = {'PRIMARY': 0, 'SECONDARY': 0, 'OTHER': 0, 'POST_HOC': 0, "OTHER_PRE_SPECIFIED": 0}  
                for measure in outcome_measures:  
                    priority = measure.get('type', '').upper()  
                    if priority in outcome_counters:  
                        outcome_counters[priority] += 1  
                    outcome_number = outcome_counters[priority]  
                    identifier = f"{priority.capitalize()} {outcome_number}"  
                    title = measure.get('title', '')  
                    description = measure.get('description', '')  
                    time_frame = measure.get('timeFrame', '')  
                    param_type = measure.get('paramType', '')  
                    dispersion_type = measure.get('dispersionType', '')  
                    unit = measure.get('unitOfMeasure', '')  
  
                    # Extracting titles from classes and converting to string
                    cate = measure.get('classes', [])  
                    titles = [cat.get("title", "") for cat in cate]  
                    titles_str = "; ".join(titles)  # Convert list of titles to a single string  
  
                    denom = measure.get('denoms', [])
                    if denom:
                        total_count = float(denom[0]['counts'][0]['value'])  # Convert total_count to float
                        
                    #def safe_float(value):
                     #   if value == 'NA':
                      #      return float('nan')  # or you could return None, or any other value that represents missing data in your context
                       # return float(value)

                    # Then modify your list comprehension:
                    #first_values = [safe_float(item['categories'][0]['measurements'][0]['value']) for item in cate]
                    #group_count = "; ".join(map(str, first_values))  # Convert list of values to a single string

                    # Calculate the sum of first_values and compare with total_count
                    #if sum(first_values) == total_count and (~pd.isna(total_count)) and (group_count!=""):
                     #   single_categorization = "yes"
                    #else:
                     #   single_categorization = "no"
                            
                    analyses = measure.get('analyses', [])  
                    if analyses:  
                        for analysis in analyses:  
                            method = analysis.get('statisticalMethod', '')  
                            a_param_type = analysis.get('paramType', '')  
  
                            outcome_data = {  
                                "Identifier": identifier,  
                                "Outcome measure": title,  
                                "Outcome description": description,  
                                "Time frame": time_frame,  
                                #"Measure type": param_type,  
                                #"Measure of Dispersion/Precision": dispersion_type,  
                                #"Unit of Measure": unit,  
                                #"Statistical Analysis Method": method,  
                                #"Outcome Analysis Parameter Type": a_param_type,  
                                "Classes": titles_str  # Use the string of titles here  
                                #"Single Categorization": single_categorization
                            }  
                            nct_data.append(outcome_data)  
                    else:  
                        outcome_data = {  
                            "Identifier": identifier,  
                            "Outcome measure": title,  
                            "Outcome description": description,  
                            "Time frame": time_frame,  
                            #"Measure type": param_type,  
                            #"Measure of Dispersion/Precision": dispersion_type,  
                            #"Unit of Measure": unit,  
                            #"Statistical Analysis Method": "",  
                            #"Outcome Analysis Parameter Type": "",  
                            "Classes": titles_str  # Use the string of titles here  
                            #"Single Categorization": single_categorization
                        }  
                        nct_data.append(outcome_data)  
            else:  
                print(f"No Outcome Measures Module found for {nct_id}")  
                nct_data.append({"error": "No Outcome Measures Module found"})  
  
            save_to_json(nct_id, nct_data)  
        except KeyError as e:  
            print(f"Error navigating JSON structure for {nct_id}: {e}")  
            save_to_json(nct_id, [{"error": f"Error navigating JSON structure: {e}"}])  
print("Data extraction complete. Results saved to individual JSON files.")  



In [ ]:
# 4. Method of aggregation

import requests  
import json  
from tqdm import tqdm  
import os  
import pandas as pd
    
# Read NCT IDs
with open('nctid_new150.txt', 'r') as file:
    nct_ids = file.read().splitlines()

output_dir = 'out_last_agg' # Specify the input file path and output directory

os.makedirs(output_dir, exist_ok=True) # Create the output directory if it doesn't exist

def save_to_json(nct_id, data):
    output_file = os.path.join(output_dir, f"{nct_id}.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

for nct_id in tqdm(nct_ids, desc="Processing NCT IDs"):  
    result = get_trial_results(nct_id)  
    if result:  
        try:  
            nct_data = []  
            study_protocol_section = result.get('resultsSection', {})  
            if 'outcomeMeasuresModule' in study_protocol_section:  
                outcome_measures = study_protocol_section['outcomeMeasuresModule'].get('outcomeMeasures', [])  
                outcome_counters = {'PRIMARY': 0, 'SECONDARY': 0, 'OTHER': 0, 'POST_HOC': 0, "OTHER_PRE_SPECIFIED": 0}  
                for measure in outcome_measures:  
                    priority = measure.get('type', '').upper()  
                    if priority in outcome_counters:  
                        outcome_counters[priority] += 1  
                    outcome_number = outcome_counters[priority]  
                    identifier = f"{priority.capitalize()} {outcome_number}"  
                    title = measure.get('title', '')  
                    description = measure.get('description', '')  
                    time_frame = measure.get('timeFrame', '')  
                    param_type = measure.get('paramType', '')  
                    dispersion_type = measure.get('dispersionType', '')  
                    unit = measure.get('unitOfMeasure', '')  
  
                    # Extracting titles from classes and converting to string
                    cate = measure.get('classes', [])  
                    titles = [cat.get("title", "") for cat in cate]  
                    titles_str = "; ".join(titles)  # Convert list of titles to a single string  
  
                    denom = measure.get('denoms', [])
                    if denom:
                        total_count = float(denom[0]['counts'][0]['value'])  # Convert total_count to float
                        
                    def safe_float(value):
                        if value == 'NA':
                            return float('nan')  # or you could return None, or any other value that represents missing data in your context
                        return float(value)

                    # Then modify your list comprehension:
                    #first_values = [safe_float(item['categories'][0]['measurements'][0]['value']) for item in cate]
                    #group_count = "; ".join(map(str, first_values))  # Convert list of values to a single string

                    # Calculate the sum of first_values and compare with total_count
                    #if sum(first_values) == total_count and (~pd.isna(total_count)) and (group_count!=""):
                     #   single_categorization = "yes"
                    #else:
                     #   single_categorization = "no"
                            
                    analyses = measure.get('analyses', [])  
                    if analyses:  
                        for analysis in analyses:  
                            method = analysis.get('statisticalMethod', '')  
                            a_param_type = analysis.get('paramType', '')  
  
                            outcome_data = {  
                                "Identifier": identifier,  
                                "Outcome measure": title,  
                                "Outcome description": description,  
                                "Time frame": time_frame,  
                                "Measure type": param_type,  
                                "Measure of Dispersion/Precision": dispersion_type,  
                                "Unit of Measure": unit,  
                                "Statistical Analysis Method": method,  
                                "Outcome Analysis Parameter Type": a_param_type,  
                                "Classes": titles_str  # Use the string of titles here  
                                #"Single Categorization": single_categorization
                            }  
                            nct_data.append(outcome_data)  
                    else:  
                        outcome_data = {  
                            "Identifier": identifier,  
                            "Outcome measure": title,  
                            "Outcome description": description,  
                            "Time frame": time_frame,  
                            "Measure type": param_type,  
                            "Measure of Dispersion/Precision": dispersion_type,  
                            "Unit of Measure": unit,  
                            "Statistical Analysis Method": "",  
                            "Outcome Analysis Parameter Type": "",  
                            "Classes": titles_str  # Use the string of titles here  
                            #"Single Categorization": single_categorization
                        }  
                        nct_data.append(outcome_data)  
            else:  
                print(f"No Outcome Measures Module found for {nct_id}")  
                nct_data.append({"error": "No Outcome Measures Module found"})  
  
            save_to_json(nct_id, nct_data)  
        except KeyError as e:  
            print(f"Error navigating JSON structure for {nct_id}: {e}")  
            save_to_json(nct_id, [{"error": f"Error navigating JSON structure: {e}"}])  
print("Data extraction complete. Results saved to individual JSON files.")  



In [ ]:
# 5. Time point (same fields as Specifc measurement)

import requests  
import json  
from tqdm import tqdm  
import os  
import pandas as pd
    
# Read NCT IDs
with open('nctid_new150.txt', 'r') as file:
    nct_ids = file.read().splitlines()

output_dir = 'out_last_time' # Specify the input file path and output directory

os.makedirs(output_dir, exist_ok=True) # Create the output directory if it doesn't exist

def save_to_json(nct_id, data):
    output_file = os.path.join(output_dir, f"{nct_id}.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

for nct_id in tqdm(nct_ids, desc="Processing NCT IDs"):  
    result = get_trial_results(nct_id)  
    if result:  
        try:  
            nct_data = []  
            study_protocol_section = result.get('resultsSection', {})  
            if 'outcomeMeasuresModule' in study_protocol_section:  
                outcome_measures = study_protocol_section['outcomeMeasuresModule'].get('outcomeMeasures', [])  
                outcome_counters = {'PRIMARY': 0, 'SECONDARY': 0, 'OTHER': 0, 'POST_HOC': 0, "OTHER_PRE_SPECIFIED": 0}  
                for measure in outcome_measures:  
                    priority = measure.get('type', '').upper()  
                    if priority in outcome_counters:  
                        outcome_counters[priority] += 1  
                    outcome_number = outcome_counters[priority]  
                    identifier = f"{priority.capitalize()} {outcome_number}"  
                    title = measure.get('title', '')  
                    description = measure.get('description', '')  
                    time_frame = measure.get('timeFrame', '')  
                    param_type = measure.get('paramType', '')  
                    dispersion_type = measure.get('dispersionType', '')  
                    unit = measure.get('unitOfMeasure', '')  
  
                    # Extracting titles from classes and converting to string
                    cate = measure.get('classes', [])  
                    titles = [cat.get("title", "") for cat in cate]  
                    titles_str = "; ".join(titles)  # Convert list of titles to a single string  
  
                    denom = measure.get('denoms', [])
                    if denom:
                        total_count = float(denom[0]['counts'][0]['value'])  # Convert total_count to float
                        
                    #def safe_float(value):
                     #   if value == 'NA':
                      #      return float('nan')  # or you could return None, or any other value that represents missing data in your context
                       # return float(value)

                    # Then modify your list comprehension:
                    #first_values = [safe_float(item['categories'][0]['measurements'][0]['value']) for item in cate]
                    #group_count = "; ".join(map(str, first_values))  # Convert list of values to a single string

                    # Calculate the sum of first_values and compare with total_count
                    #if sum(first_values) == total_count and (~pd.isna(total_count)) and (group_count!="") and (~pd.isna(sum(first_values))):
                    #    single_categorization = "yes"
                    #else:
                    #    single_categorization = "no"
                            
                    analyses = measure.get('analyses', [])  
                    if analyses:  
                        for analysis in analyses:  
                            method = analysis.get('statisticalMethod', '')  
                            a_param_type = analysis.get('paramType', '')  
  
                            outcome_data = {  
                                "Identifier": identifier,  
                                "Outcome measure": title,  
                                "Outcome description": description,  
                                "Time frame": time_frame,  
                                #"Measure type": param_type,  
                                #"Measure of Dispersion/Precision": dispersion_type,  
                                #"Unit of Measure": unit,  
                                #"Statistical Analysis Method": method,  
                                #"Outcome Analysis Parameter Type": a_param_type,  
                                "Classes": titles_str  # Use the string of titles here  
                                #"Single Categorization": single_categorization
                            }  
                            nct_data.append(outcome_data)  
                    else:  
                        outcome_data = {  
                            "Identifier": identifier,  
                            "Outcome measure": title,  
                            "Outcome description": description,  
                            "Time frame": time_frame,  
                            #"Measure type": param_type,  
                            #"Measure of Dispersion/Precision": dispersion_type,  
                            #"Unit of Measure": unit,  
                            #"Statistical Analysis Method": "",  
                            #"Outcome Analysis Parameter Type": "",  
                            "Classes": titles_str  # Use the string of titles here  
                            #"Single Categorization": single_categorization
                        }  
                        nct_data.append(outcome_data)  
            else:  
                print(f"No Outcome Measures Module found for {nct_id}")  
                nct_data.append({"error": "No Outcome Measures Module found"})  
  
            save_to_json(nct_id, nct_data)  
        except KeyError as e:  
            print(f"Error navigating JSON structure for {nct_id}: {e}")  
            save_to_json(nct_id, [{"error": f"Error navigating JSON structure: {e}"}])  
print("Data extraction complete. Results saved to individual JSON files.")  

